In [1]:
using ITensors
using ITensorMPS
using PolyChaos
using LinearAlgebra
using Plots
using Observers #for TDVP

In [3]:
function spin_operators(M)

    # Build sparse matrix version of basic spin (Pauli) operators :
    sp = spdiagm(2,2,1=>ones(1))
    sm = spdiagm(2,2,-1=>ones(1))
    sz = spdiagm(2,2,0=>[1;-1]);
    num = spdiagm(2,2,0=>[0;1])
    # Notice there are NO factors of (1/2) for spin-1/2 included here.

    # Construct spin operators for each spin in the full Hilbert space :
    Sz = Vector{Any}(undef, M)
    Sp = Vector{Any}(undef, M)
    Sm = Vector{Any}(undef, M)
    Num = Vector{Any}(undef,M)
    for m=1:M
        Sz[m] = kronecker(kronecker(spdiagm(2^(m-1),2^(m-1),0=>ones(2^(m-1))),sz),spdiagm(2^(M-m),2^(M-m),0=>ones(2^(M-m))));
        Sp[m] = kronecker(kronecker(spdiagm(2^(m-1),2^(m-1),0=>ones(2^(m-1))),sp),spdiagm(2^(M-m),2^(M-m),0=>ones(2^(M-m))));
        Sm[m] = kronecker(kronecker(spdiagm(2^(m-1),2^(m-1),0=>ones(2^(m-1))),sm),spdiagm(2^(M-m),2^(M-m),0=>ones(2^(M-m))));
        Num[m] = kronecker(kronecker(spdiagm(2^(m-1),2^(m-1),0=>ones(2^(m-1))),num),spdiagm(2^(M-m),2^(M-m),0=>ones(2^(M-m))));
    end
    return Sz,Sp,Sm,Num
end
function matrix_operators(M)

    Sz,Sp,Sm,_ = spin_operators(M)
    cdag_mat = Vector{Any}(undef,M)
    c_mat = Vector{Any}(undef,M)

    for n=1:M
        #Build JW_string
        Z = JW_string_mat(Sz,n,M)
        cdag_mat[n] = Z*Sm[n]
        c_mat[n]  = Z*Sp[n]
    end
    return cdag_mat,c_mat
   
end
function JW_string_mat(Sz,site,M;kwargs...)
    inds = get(kwargs,:inds,1:(site-1))

    Z = 1.0*Matrix(I, 2^M, 2^M)
    for i in inds
        Z = Z*Sz[i];
    end
    return Z
end
function calculate_ρ_from_correlation_matrix(C, mode_subset; eps=1e-14,kwargs...)
    """
        gaussian_rdm_density_matrix(C, mode_subset,qA; eps=1e-14)

    Construct the fermionic Gaussian density matrix

        ρ = ⊗ₖ [(1-νₖ)|0><0| + νₖ|1><1|]

    from the correlation matrix C restricted to the mode_subset modes.

    Arguments
    ---------
    C            : correlation matrix
    mode_subset  : modes to calculate rdm
    qA           : ancilla modes

    Returns
    -------
    ρ      : Dense many-body density matrix
    """

    N = length(mode_subset)
    ddag,d = matrix_operators(N)
    dim = size(ddag[1],1)

    #take the subset and symmetrize
    C = transpose(C[mode_subset,mode_subset])
    C = (C + C') / 2

    # 2. Diagonalize correlation matrix
    eig = eigen(Hermitian(C))
    ν = clamp.(real(eig.values), eps, 1-eps)
    U = eig.vectors

    # Construct rotated fermion operators
    #    f_k = Σ_i U†_{ki} d_i
    f = Vector{Any}(undef, N)
    fdag = Vector{Any}(undef, N)

    for k in 1:N
        fk = zero(d[1])
        for i in 1:N
            fk += conj(U[i,k]) * d[i]
        end
        f[k] = fk
        fdag[k] = fk'
    end

    # Construct many-body density matrix
    #    ρ = Π_k [(1-ν_k)(1-n_k) + ν_k n_k]
    # where n_k = f†_k f_k

    ρ = Matrix(I, dim, dim)
    for k in 1:N
        nk = fdag[k] * f[k]
        ρk = (1-ν[k]) * (Matrix(I, dim, dim) - nk) +
              ν[k] * nk
        ρ *= ρk
    end

    ##checks it's a valid density matrix up to a tolerance 1e-5
    ρ_test(ρ,1e-5) 

    return ρ
end
function calculate_Λ_from_correlation_matrix(C, mode_subset,qA; eps=1e-14,kwargs...)
    """
        gaussian_rdm_density_matrix(C, mode_subset,qA; eps=1e-14)

    Construct the fermionic Gaussian density matrix

        ρ = ⊗ₖ [(1-νₖ)|0><0| + νₖ|1><1|]

    from the correlation matrix C restricted to the mode_subset modes.

    Arguments
    ---------
    C            : correlation matrix
    mode_subset  : modes to calculate rdm
    qA           : ancilla modes

    Returns
    -------
    ρ      : Dense many-body density matrix
    """

    symmetry_subspace = get(kwargs,:symmetry_subspace, "Number conserving")
    N = length(mode_subset)
    ddag,d = matrix_operators(N)
    dim = size(ddag[1],1)

    #take the subset and symmetrize
    C = transpose(C[mode_subset,mode_subset])
    C = (C + C') / 2


    # 2. Diagonalize correlation matrix
    eig = eigen(Hermitian(C))
    ν = clamp.(real(eig.values), eps, 1-eps)
    U = eig.vectors

    # Construct rotated fermion operators
    #    f_k = Σ_i U†_{ki} d_i
    f = Vector{Any}(undef, N)
    fdag = Vector{Any}(undef, N)

    for k in 1:N
        fk = zero(d[1])
        for i in 1:N
            fk += conj(U[i,k]) * d[i]
        end
        f[k] = fk
        fdag[k] = fk'
    end

    # Construct many-body density matrix
    #    ρ = Π_k [(1-ν_k)(1-n_k) + ν_k n_k]
    # where n_k = f†_k f_k

    ρ = Matrix(I, dim, dim)
    for k in 1:N
        nk = fdag[k] * f[k]
        ρk = (1-ν[k]) * (Matrix(I, dim, dim) - nk) +
              ν[k] * nk

        ρ *= ρk
    end

    # gates =  ancilla_phase_gate_swap(qA,ddag,d)
    # ρ = apply_gates_to_ρmat(ρ,gates)

    ##PH transform
    for index in qA
  #      gate = Sp[index] + Sm[index]
        gate = ddag[index] +d[index]#+ ((-1)^(N_ring))*c_mat[index]
        ρ = gate*ρ*gate'
    end

    ##checks it's a valid density matrix up to a tolerance 1e-5
    ρ_test(ρ,1e-5) 

    ##reshape to give the map
    Λ = ρ_to_Λ(ρ,N_ring) 
    if symmetry_subspace =="Number conserving"
        qN = extract_physical_modes(N_ring)
        Λ = Λ[qN,qN]
    end

    return Λ
end

JW_string_mat (generic function with 1 method)

In [19]:
Base.@kwdef struct BathParameters
    Γ::Float64 #Coupling to system
    β::Float64 #inverse temperature
    μ::Float64 #chemical potential
    D::Float64 #bandwidth
    N::Int     #Number of chain modes
end

Base.@kwdef struct SystemParameters
    B::Float64                  #B field
    J::Float64                  #hoppings
    N_ring::Int                 #size of ring
    occupations::Vector{Float64} # Vector of initial occupations
    compute_maps_bool::Bool     #Boolean on whether to calculate maps
end

abstract type ThermofieldSector end
struct Filled <: ThermofieldSector end
struct Empty <: ThermofieldSector end

struct ChainLayout
    system
    filled_bath
    empty_bath
end

"""
Initialisation functions
"""
fermi_factor(ω,β,μ) = 1 / (exp(β*(ω-μ)) + 1)
heaviside(t) = 0.5 * (sign.(t) .+ 1)

function semicircular_density(Γ,ω,D)
    J = real((2*Γ/(π^2))*sqrt.(Complex.(1 .-(ω/D).^2)))
    return J

end

function box_spectral_density(Γ,ω,D)
    #Box spectral density
    return J = (Γ/(2*π))*(heaviside(ω .+ D) .- heaviside(ω .- D))
end

function thermofield_spectral_density(ω,bath::BathParameters,::Filled)
    #Spectral density for a filled chain
    J = box_spectral_density(bath.Γ,ω,bath.D)
   # J = semicircular_density(bath.Γ,ω,bath.D)
    return J * fermi_factor(ω,bath.β,bath.μ)
end

function thermofield_spectral_density(ω,bath::BathParameters,::Empty)
    #Spectral density for an empty chain
    J = box_spectral_density(bath.Γ,ω,bath.D)
  #  J = semicircular_density(bath.Γ,ω,bath.D)
    return J * (1 - fermi_factor(ω,bath.β,bath.μ))
end

function chain_mapping(bath::BathParameters,sector::ThermofieldSector)

    #Calculates the chain coefficients using PolyChaos.jl

    spec_fun(ω) = thermofield_spectral_density(ω,bath,sector)

    #support needs to be larger than spectral function for numerical reasons.
    support = (-2*bath.D,2*bath.D)

    meas = Measure("thermofield",spec_fun,support,false,Dict())
    op = OrthoPoly("chain",bath.N-1,meas;Nquad=100000)

    α = coeffs(op)[:,1]
    β = coeffs(op)[:,2]

    return α,sqrt.(β)
end

function ChainLayout(N_bath,Nsys)
    ##Arranges the chains in interleaved fashion, with the system at the start. Any
    ##other layout can be encoded here and will follow through to the rest of the code.


    N = 2*N_bath+Nsys

    ChainLayout(
        1:Nsys,
        Nsys+1:2:N,
        Nsys+2:2:N,
    )
end

function add_chain(H,inds,energies,hoppings)
    ##Adds MPO terms and associated single particle hamiltonian elements 
    ##for the thermofield chain

    N = length(inds)
    for i in 1:N
        #os += energies[i],"N",inds[i]

        H[inds[i],inds[i]] = energies[i]

        if i < N
            t = hoppings[i]
         #   os += t,"Cdag",inds[i],"C",inds[i+1]
          #  os += t,"Cdag",inds[i+1],"C",inds[i]

            H[inds[i],inds[i+1]] = t
            H[inds[i+1],inds[i]] = t
        end
    end
    return H#,os
end
function couple_sites(H,i,j,t)
    #Couples two sites, used for the system-chain coupling

   # os += t,"Cdag",i,"C",j
    #os += t,"Cdag",j,"C",i

    H[i,j] = t
    H[j,i] = t
    return H#,os
end

function initialise_setup(bath,sys)
    #builds single particle hamiltonian and single particle correlation matrix

    if sys.compute_maps_bool
        Nsys = 2*sys.N_ring
        qS = 1:sys.N_ring
        qA = sys.N_ring+1:2*N_ring
    else
        Nsys = sys.N_ring
        qS = 1:sys.N_ring
        qA = 0:0
    end

    layout =ChainLayout(bath.N,Nsys)

    N = 2*bath.N+Nsys
    Hsingle = zeros(ComplexF64,N,N)
    initial_correlation_matrix = zeros(ComplexF64,N,N)

    #chain coefficients for the two chains
    εF,tF =chain_mapping(bath,Filled())
    εE,tE =chain_mapping(bath,Empty())

    #adds the terms for the chains
    Hsingle = add_chain(Hsingle,layout.filled_bath,εF,tF[2:end])
    Hsingle = add_chain(Hsingle,layout.empty_bath,εE,tE[2:end])

    #
    # system
    #

    #Create system hamiltonian terms
        for i =1:sys.N_ring
            Hsingle[qS[i],qS[i]] = -2*sys.B
            if i <sys.N_ring
                Hsingle[qS[i+1],qS[i]] = -2*sys.J
                Hsingle[qS[i],qS[i+1]] = -2*sys.J
            elseif i!= 1
                @assert(i==sys.N_ring)
                Hsingle[qS[1],qS[i]] = -2*sys.J
                Hsingle[qS[i],qS[1]] = -2*sys.J
            end
        end

    #
    # bath-system couplings
    #

    Hsingle = couple_sites(Hsingle,first(layout.filled_bath),last(layout.system),first(tF))
    Hsingle = couple_sites(Hsingle,first(layout.empty_bath),last(layout.system),first(tE))

     #create initial correlation matrix
    [initial_correlation_matrix[i,i] = 1 for i in layout.filled_bath]
    if sys.compute_maps_bool
        for i =1:sys.N_ring 
            initial_correlation_matrix[qS[i],qS[i]] = 0.5
            initial_correlation_matrix[qS[i],qA[i]] = 0.5
            initial_correlation_matrix[qA[i],qS[i]] = 0.5
            initial_correlation_matrix[qA[i],qA[i]] = 0.5
        end
    else
        for i =1:sys.N_ring 
            initial_correlation_matrix[qS[i],qS[i]] = sys.occupations[i]
        end
    end

    return Hsingle,initial_correlation_matrix
end


bath = BathParameters(Γ = 0.5,β = 10.0,μ = 0.0,D = 1.0,N = 20)

system = SystemParameters(B = 1,J = 1,N_ring = 3,occupations=[0.2,0.8,0.1], compute_maps_bool = false)

Hsingle,initial_correlation_matrix = initialise_setup(bath,system);
